In [ ]:
import numpy as np
import pandas as pd
from statsmodels.api import Logit 
import requests
import duckdb
import os
import s3fs
import sys
sys.path.append('../outils')

import fonctions_nomenclature as fnom
import fonctions_preparation_table as fprep
%load_ext autoreload
%autoreload 2

np.random.seed(123)

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})
BUCKET = "mathildedonnenfeld/diffusion/bistrots_et_bulletins"


FILE_KEY_S3 = "table_analyse_muni.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    df = pd.read_csv(file_in, sep=",")

df.head()   

In [ ]:
#Provisoire, le temps que la table soit enregistrée dans le bucket

df_sous_ech = df[df['POPULATION'] > 3500]

In [ ]:
df_sous_ech = df_sous_ech.drop(columns =['Unnamed: 0.1', 'Unnamed: 0'])

In [ ]:
df_sous_ech.columns

In [ ]:
# Préparation des variables 

df_sous_ech['ind1']= df_sous_ech['ind1'].astype("category")

In [ ]:
## A SUPPRIMER QUAND L ISSUE ASSOCIEE EST REGLEE

# Nombre de valeurs manquantes
print("Nombre de NA :", df_sous_ech['nb_etablissements'].isna().sum())
# Afficher les observations où nb_etablissements est manquant
df_NA = df_sous_ech[
    df_sous_ech['nb_etablissements'].isna()
]

df_NA.head()

#Pour le moment, supprimer ces lignes 
df_sous_ech = df_sous_ech.dropna(subset=['nb_etablissements'])

In [ ]:
#######  A RETIRER QUAND L ISSUE EST REGLEE

# Nombre de valeurs manquantes
print("Nombre de NA :", df_sous_ech['ind1'].isna().sum())
# Afficher les observations où nb_etablissements est manquant
df_NA = df_sous_ech[
    df_sous_ech['ind1'].isna()
]

df_NA.head()

#Pour le moment, supprimer ces lignes 
df_sous_ech = df_sous_ech.dropna(subset=['ind1'])

## Déterminants du vote de droite 

In [ ]:
# Regression logistique simple
X = df_sous_ech[['nb_etablissements']]
y = df_sous_ech['ind1']

# Fit OLS model
lr = Logit(endog=y,exog=X) 
res = lr.fit() 

In [ ]:
#résumé des résultats 
print(res.summary()) 

# Le coefficient est négatif, et la pvaleur <0.05. 
# L'ajout d'un bistrot est associée à une probabilité moins élevée que la liste élue soit de droite. 

In [ ]:
df_sous_ech.columns

In [ ]:
print(df_sous_ech[['POPULATION', 'NIVVIE_MEDIAN', 'TAUX_PAUVRETE',
       'nb_etablissements', 'age_moyen_bistrots','Nombre candidats', 'Abstentions']].dtypes)

In [ ]:
# Régression linéaire multiple 

## Matrice de variance covariance pour reprérer les corrélations 
cov_matrix = np.cov(df_sous_ech[['POPULATION', 'NIVVIE_MEDIAN', 'TAUX_PAUVRETE',
       'nb_etablissements', 'age_moyen_bistrots','Nombre candidats', 'Abstentions']], ddof=1)


## Déterminants du vote RN